In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 0. PREPRARE

In [2]:
!pip install fire
!pip install unidecode
#!pip install librosa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.4/88.4 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fire: filename=fire-0.6.0-py2.py3-none-any.whl size=117029 sha256=d9df4182ec4d3d8f93518c30712952ced21ed622d7600c3241aaf8f0239ea72b
  Stored in directory: /root/.cache/pip/wheels/d6/6d/5d/5b73fa0f46d01a793713f8859201361e9e581ced8c75e5c6a3
Successfully built fire
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.5/235.5 kB 5.4 MB/s eta 0:00:00


In [3]:
#!wget https://openaipublic.azureedge.net/jukebox/models/5b/vqvae.pth.tar
#!wget https://openaipublic.azureedge.net/jukebox/models/5b/prior_level_2.pth.tar

In [4]:
import sys

sys.path.append('/content/drive/MyDrive/Modeling/JukeMIR')

In [5]:
import pathlib
from argparse import ArgumentParser

# imports and set up Jukebox's multi-GPU parallelization
import jukebox
from jukebox.hparams import Hyperparams, setup_hparams
from jukebox.make_models import MODELS, make_prior, make_vqvae
from jukebox.utils.dist_utils import setup_dist_from_mpi
from tqdm import tqdm

import librosa as lr
import numpy as np
import torch
import os
import pickle

### 1. PARAMS

In [6]:
# --- MODEL PARAMS ---

JUKEBOX_SAMPLE_RATE = 44100
T = 8192
SAMPLE_LENGTH = 1048576  # ~23.77s, which is the param in JukeMIR
DEPTH = 36

In [7]:
def load_audio_from_file(fpath):
    audio, _ = lr.load(fpath, sr=JUKEBOX_SAMPLE_RATE)
    if audio.ndim == 1:
        audio = audio[np.newaxis]
    audio = audio.mean(axis=0)

    # normalize audio
    norm_factor = np.abs(audio).max()
    if norm_factor > 0:
        audio /= norm_factor

    return audio.flatten()

In [8]:
def split_audio_into_chunks(audio, chunk_size=SAMPLE_LENGTH, sample_rate=JUKEBOX_SAMPLE_RATE):

    """
    Splits the audio into chunks of `chunk_size` with `sample_rate`, padding the last chunk with zeros if necessary.

    Args:
    - audio (numpy.ndarray): The loaded and preprocessed audio signal.
    - chunk_size (int): Size of each chunk in number of samples. Default is set for ~23.77 seconds.
    - sample_rate (int): Sampling rate of the audio. Default is 44100 Hz.

    Returns:
    - List of audio chunks (numpy.ndarray), with the last chunk padded with zeros if shorter than `chunk_size`.
    """

    num_samples = audio.shape[0]
    # Calculate the total number of chunks, including any partial chunk at the end
    total_chunks = int(np.ceil(num_samples / chunk_size))
    chunks = []

    for i in range(total_chunks):
        start_sample = i * chunk_size
        end_sample = start_sample + chunk_size
        if end_sample > num_samples:
            # If the chunk extends beyond the length of the audio, pad it with zeros
            pad_length = end_sample - num_samples
            chunk = np.concatenate((audio[start_sample:num_samples], np.zeros(pad_length)))
        else:
            chunk = audio[start_sample:end_sample]
        chunks.append(chunk)

    return chunks

In [9]:
import numpy as np

def split_audio_into_four_random_parts(audio, chunk_size=SAMPLE_LENGTH, sample_rate=JUKEBOX_SAMPLE_RATE):
    """
    Splits the audio into four equal parts and randomly samples `chunk_size` from each part,
    ensuring that the sampled chunk does not exceed the part's boundary.

    Args:
    - audio (numpy.ndarray): The loaded and preprocessed audio signal.
    - chunk_size (int): Size of each sample chunk in number of samples.
    - sample_rate (int): Sampling rate of the audio. Default is 44100 Hz.

    Returns:
    - List of four audio chunks (numpy.ndarray), each randomly sampled from one of the four equal parts of the audio.
    """

    num_samples = audio.shape[0]
    part_size = num_samples // 4  # Divide the audio into four equal parts

    chunks = []
    for i in range(4):
        # Calculate start and end samples for each part
        start_of_part = i * part_size
        end_of_part = start_of_part + part_size

        # Ensure the sampled chunk does not exceed the part's boundary
        max_start_sample = end_of_part - chunk_size

        # Randomly choose start sample within the part
        start_sample = np.random.randint(start_of_part, max_start_sample)
        chunk = audio[start_sample:start_sample + chunk_size]


        chunks.append(chunk)

    return chunks

In [10]:
# # For Test

# audio_path = "/content/drive/MyDrive/datasets/music/1657.wav"

# # MONTERO - Lil Nas X
# # 2m 17s -> 137s
# audio = load_audio_from_file(audio_path)
# print("Audio Shape : ", audio.shape)

# audio_chunks = split_audio_into_chunks(audio)
# print("Length of Audio Chunks : ", len(audio_chunks))
# print("Audio Chunk Shape : ", audio_chunks[5].shape)

### 2-1. JukeMIR total

In [ ]:
def audio_padding(audio, target_length):
    padding_length = target_length - audio.shape[0]
    padding_vector = np.zeros(padding_length)
    padded_audio = np.concatenate([audio, padding_vector], axis=0)
    return padded_audio

In [ ]:
def get_z(audio, vqvae):
    audio = audio[: JUKEBOX_SAMPLE_RATE * 25]

    zs = vqvae.encode(torch.cuda.FloatTensor(audio[np.newaxis, :, np.newaxis]))

    z = zs[-1].flatten()[np.newaxis, :]

    if z.shape[-1] < 8192:
        raise ValueError("Audio file is not long enough")

    return z


def get_cond(hps, top_prior):
    sample_length_in_seconds = 62

    hps.sample_length = (int(sample_length_in_seconds * hps.sr) // top_prior.raw_to_tokens
                        ) * top_prior.raw_to_tokens

    # NOTE: the 'lyrics' parameter is required, which is why it is included,
    # but it doesn't actually change anything about the `x_cond`, `y_cond`,
    # nor the `prime` variables
    metas = [dict(artist="unknown", genre="unknown", total_length=hps.sample_length,
                  offset=0, lyrics="unknown")] * hps.n_samples

    labels = [None, None, top_prior.labeller.get_batch_labels(metas, "cuda")]

    x_cond, y_cond, prime = top_prior.get_cond(None, top_prior.get_y(labels[-1], 0))

    x_cond = x_cond[0, :T][np.newaxis, ...]
    y_cond = y_cond[0][np.newaxis, ...]

    return x_cond, y_cond


def get_final_activations(z, x_cond, y_cond, top_prior):
    x = z[:, :T]

    # make sure that we get the activations
    top_prior.prior.only_encode = True

    # encoder_kv and fp16 are set to the defaults, but explicitly so
    out = top_prior.prior.forward(
        x, x_cond=x_cond, y_cond=y_cond, encoder_kv=None, fp16=False
    )

    return out


def get_acts_from_file(fpath, hps, vqvae, top_prior, meanpool):

    audio_entire = load_audio_from_file(fpath)
    audio_sets = split_audio_into_chunks(audio_entire)

    acts_set = []

    for audio in audio_sets:
      # zero padding
      if audio.shape[0] < SAMPLE_LENGTH:
          audio = audio_padding(audio, SAMPLE_LENGTH)

      # run vq-vae on the audio
      z = get_z(audio, vqvae)

      # get conditioning info
      x_cond, y_cond = get_cond(hps, top_prior)

      # get the activations from the LM
      acts = get_final_activations(z, x_cond, y_cond, top_prior)

      # postprocessing
      acts = acts.squeeze().type(torch.float32)  # shape: (8192, 4800)

      # average mode
      assert acts.size(0) % meanpool == 0, "The 'meanpool' param must be divisible by 8192. "
      acts = acts.view(meanpool, -1, 4800)
      acts = acts.mean(dim=1)
      acts = np.array(acts.cpu())

      acts_set.append(acts)

    return acts_set

### 2-2. JukeMIR - sampling

In [11]:
def audio_padding(audio, target_length):
    padding_length = target_length - audio.shape[0]
    padding_vector = np.zeros(padding_length)
    padded_audio = np.concatenate([audio, padding_vector], axis=0)
    return padded_audio

In [12]:
def get_z(audio, vqvae):
    audio = audio[: JUKEBOX_SAMPLE_RATE * 25]

    zs = vqvae.encode(torch.cuda.FloatTensor(audio[np.newaxis, :, np.newaxis]))

    z = zs[-1].flatten()[np.newaxis, :]

    if z.shape[-1] < 8192:
        raise ValueError("Audio file is not long enough")

    return z


def get_cond(hps, top_prior):
    sample_length_in_seconds = 62

    hps.sample_length = (int(sample_length_in_seconds * hps.sr) // top_prior.raw_to_tokens
                        ) * top_prior.raw_to_tokens

    # NOTE: the 'lyrics' parameter is required, which is why it is included,
    # but it doesn't actually change anything about the `x_cond`, `y_cond`,
    # nor the `prime` variables
    metas = [dict(artist="unknown", genre="unknown", total_length=hps.sample_length,
                  offset=0, lyrics="unknown")] * hps.n_samples

    labels = [None, None, top_prior.labeller.get_batch_labels(metas, "cuda")]

    x_cond, y_cond, prime = top_prior.get_cond(None, top_prior.get_y(labels[-1], 0))

    x_cond = x_cond[0, :T][np.newaxis, ...]
    y_cond = y_cond[0][np.newaxis, ...]

    return x_cond, y_cond


def get_final_activations(z, x_cond, y_cond, top_prior):
    x = z[:, :T]

    # make sure that we get the activations
    top_prior.prior.only_encode = True

    # encoder_kv and fp16 are set to the defaults, but explicitly so
    out = top_prior.prior.forward(
        x, x_cond=x_cond, y_cond=y_cond, encoder_kv=None, fp16=False
    )

    return out


def get_acts_from_file(audio_sets, hps, vqvae, top_prior, meanpool):

    acts_set = []

    for audio in audio_sets:
      # zero padding
      if audio.shape[0] < SAMPLE_LENGTH:
          audio = audio_padding(audio, SAMPLE_LENGTH)

      # run vq-vae on the audio
      z = get_z(audio, vqvae)

      # get conditioning info
      x_cond, y_cond = get_cond(hps, top_prior)

      # get the activations from the LM
      acts = get_final_activations(z, x_cond, y_cond, top_prior)

      # postprocessing
      acts = acts.squeeze().type(torch.float32)  # shape: (8192, 4800)

      # average mode
      assert acts.size(0) % meanpool == 0, "The 'meanpool' param must be divisible by 8192. "
      acts = acts.view(meanpool, -1, 4800)
      acts = acts.mean(dim=1)
      acts = acts[32, :]
      acts = np.array(acts.cpu())

      acts_set.append(acts)

    return acts_set

### 3. Hyperparams

In [13]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
VQVAE_MODELPATH = "/content/drive/MyDrive/Modeling/JukeMIR/pretrained/vqvae.pth.tar"
PRIOR_MODELPATH = "/content/drive/MyDrive/Modeling/JukeMIR/pretrained/prior_level_2.pth.tar"

INPUT_DIR = r"/content/drive/MyDrive/Modeling/datasets/music/1658.wav"
OUTPUT_DIR = r"/content/drive/MyDrive/JukeMIR/Modeling/sample_output/1658.npy"

AVERAGE_SLICES = 64  # For average pooling. "1" means average all frames.
#  Since the output shape is 8192 * 4800, the params bust can divide 8192.
USING_CACHED_FILE = False
model = "5b"  # might not fit to other settings, e.g., "1b_lyrics" or "5b_lyrics"

In [14]:
# --- SETTINGS ---
input_path = pathlib.Path(INPUT_DIR)
output_path = pathlib.Path(OUTPUT_DIR)
device = DEVICE

In [15]:
# Set up VQVAE

hps = Hyperparams()
hps.sr = 44100
hps.n_samples = 8
hps.name = "samples"
chunk_size = 32
max_batch_size = 16
hps.levels = 3
hps.hop_fraction = [0.5, 0.5, 0.125]
vqvae, *priors = MODELS[model]
hps_1 = setup_hparams(vqvae, dict(sample_length=SAMPLE_LENGTH))
hps_1.restore_vqvae = VQVAE_MODELPATH
vqvae = make_vqvae(hps_1, device)

Restored from /content/drive/MyDrive/Modeling/JukeMIR/pretrained/vqvae.pth.tar
Loading vqvae in eval mode


In [16]:
# Set up language model
hps_2 = setup_hparams(priors[-1], dict())
hps_2["prior_depth"] = DEPTH
hps_2.restore_prior = PRIOR_MODELPATH
top_prior = make_prior(hps_2, vqvae, device)

Loading artist IDs from /content/drive/MyDrive/Modeling/JukeMIR/jukebox/data/ids/v2_artist_ids.txt
Loading artist IDs from /content/drive/MyDrive/Modeling/JukeMIR/jukebox/data/ids/v2_genre_ids.txt
Level:2, Cond downsample:None, Raw to tokens:128, Sample length:1048576
Converting to fp16 params
Restored from /content/drive/MyDrive/Modeling/JukeMIR/pretrained/prior_level_2.pth.tar
Loading prior in eval mode


### 4. Representation

In [17]:
with open('/content/drive/MyDrive/Modeling/datasets/music_lr/1300_1900.pkl', 'rb') as f:
  sample1300_1900 = pickle.load(f)

In [18]:
def process_all_audio(audio_sets, hps, vqvae, top_prior, meanpool):

    outputs= {}
    # 각 오디오 파일에 대해 반복 처리합니다.
    for i, (key, audio) in enumerate(audio_sets.items()):

        # 표현을 추출하는 코드를 실행합니다.
        with torch.no_grad():
            representation = get_acts_from_file(
                audio, hps, vqvae, top_prior, meanpool=AVERAGE_SLICES
            )

        # 결과를 npy 파일로 저장합니다.
        representation = np.array(representation)
        representation = representation.squeeze()
        outputs[key] = representation

        if i% 10 == 0:
          print(f"{i}th is done")

    return outputs

In [19]:
outputs = process_all_audio(sample1300_1900, hps, vqvae, top_prior, meanpool=AVERAGE_SLICES)

<ipython-input-12-6c5e71f96ea7>:4: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:83.)
  zs = vqvae.encode(torch.cuda.FloatTensor(audio[np.newaxis, :, np.newaxis]))


0th is done
10th is done
20th is done
30th is done
40th is done
50th is done
60th is done
70th is done
80th is done
90th is done
100th is done
110th is done
120th is done
130th is done
140th is done
150th is done
160th is done
170th is done
180th is done
190th is done
200th is done
210th is done
220th is done
230th is done
240th is done
250th is done
260th is done
270th is done
280th is done
290th is done
300th is done
310th is done
320th is done
330th is done
340th is done
350th is done
360th is done
370th is done
380th is done
390th is done
400th is done
410th is done
420th is done
430th is done
440th is done
450th is done
460th is done
470th is done
480th is done
490th is done
500th is done
510th is done
520th is done
530th is done
540th is done
550th is done
560th is done
570th is done
580th is done
590th is done


In [20]:
pickle_filename = f'/content/drive/MyDrive/Modeling/finaldata1300_1900.pkl'
with open(pickle_filename, 'wb') as f:
  pickle.dump(outputs, f)

In [ ]:
filenames = os.listdir('/content/drive/MyDrive/datasets/music')
audio_files = [f for f in filenames if f.endswith('.wav')]
sorted_filenames = sorted(audio_files, key=lambda x: int(x.split('.')[0]))

In [ ]:
def process_and_save_all_audio(startindex, lastindex, audio_files, hps, vqvae, top_prior, meanpool):

    outputs= {}
    # 각 오디오 파일에 대해 반복 처리합니다.
    for i, audio_file in enumerate(audio_files[startindex:lastindex]):
        filename = int(audio_file.split('.')[0])
        input_path = os.path.join('/content/drive/MyDrive/datasets/music', audio_file)

        # 표현을 추출하는 코드를 실행합니다.
        with torch.no_grad():
            representation = get_acts_from_file(
                input_path, hps, vqvae, top_prior, meanpool=AVERAGE_SLICES
            )

        # 결과를 npy 파일로 저장합니다.
        representation = np.array(representation)
        representation = representation.squeeze()
        outputs[filename] = representation

        if i% 10 == 0:
          print(f"{filename} is done")

    pickle_filename = f'/content/drive/MyDrive/datasets/MIR/{startindex}_{lastindex}.pkl'
    with open(pickle_filename, 'wb') as f:
      pickle.dump(outputs, f)

    return outputs

In [ ]:
start = 1600
end = 2076
print(f"------------start : {start} | end : {end-1}----------------")
ouptut = process_and_save_all_audio(start, end, sorted_filenames, hps, vqvae, top_prior, meanpool=AVERAGE_SLICES)

------------start : 1600 | end : 2075----------------
1600 is done
1610 is done
1620 is done
1630 is done
1640 is done
1650 is done
1660 is done
1670 is done
1680 is done
1690 is done
1700 is done
1710 is done
1720 is done
1730 is done
1740 is done
1750 is done
1760 is done
1770 is done
1780 is done
1790 is done
1800 is done
1810 is done
1820 is done
1830 is done
1840 is done
1850 is done
1860 is done
1870 is done
1880 is done
1890 is done
1900 is done
1910 is done
1920 is done
1930 is done
1940 is done
1950 is done
1960 is done
1970 is done
1980 is done
1990 is done
2000 is done
2010 is done
2020 is done
2030 is done
2040 is done
2050 is done
2060 is done
2070 is done


In [ ]:
with open(f'/content/drive/MyDrive/datasets/MIR/1600_2076.pkl', 'rb') as f:
    loaded_outputs = pickle.load(f)

In [ ]:
value_list = [100, 600, 1100, 1600]
max_end = 2076

for start in value_list:
  end = min(start+500, max_end)
  print(f"------------start : {start} | end : {end-1}----------------")
  ouptut = process_and_save_all_audio(start, end, sorted_filenames, hps, vqvae, top_prior, meanpool=AVERAGE_SLICES)

In [ ]:
with open(f'/content/drive/MyDrive/drive atasets/MIR/0_100.pkl', 'rb') as f:
    loaded_outputs = pickle.load(f)